# Verified local sandbox evaluation

This notebook uses actual Linux Docker execution, real LangGraph/Pydantic/SQLite, and deterministic offline reasoning. The optimizer fault-injection examples alter the grader only; learner execution is never mocked. No OpenAI inference is claimed.

In [ ]:
from pathlib import Path
import os, sys, tempfile, json
workspace=Path.cwd()
if not (workspace/'src').exists():workspace=workspace.parent
sys.path.insert(0,str(workspace))
run_dir=tempfile.TemporaryDirectory(prefix='sprints-verified-notebook-')
os.environ.update(SPRINTS_LLM='offline',SPRINTS_EMBEDDINGS='hashing',SPRINTS_LIVE_AUTHORIZED='0',SPRINTS_DB_PATH=str(Path(run_dir.name)/'learners.db'),SPRINTS_CHROMA_DIR=str(Path(run_dir.name)/'chroma'),ANONYMIZED_TELEMETRY='False',LANGSMITH_TRACING='false',LANGCHAIN_TRACING_V2='false')
from src.auth import MockIdentityProvider
from src.service import OrchestratorService
from src.main import BUGGY,FIXED
from src.tools.code_repl import sandbox_capabilities
from src.tools.db_queries import build_database,get_learner_profile
capability=sandbox_capabilities()
print(json.dumps(capability,indent=2))
assert capability['available'], 'Verified Linux sandbox is required; no fixture substitution'
build_database()
identity=MockIdentityProvider()
session=identity.issue_mock_session('learner-1005')

## Actual container grading
Both submissions go through the unchanged five-second isolated executor. A timeout or infrastructure failure remains a failure; no invented grade substitutes for it.

In [ ]:
app=OrchestratorService(identity,warm=False)
actual=[]
for submission in (BUGGY,FIXED):
    result=app.ask(session,{'text':submission})
    actual.append(result)
    print(json.dumps(result,indent=2))
assert all(r['status']=='graded' for r in actual)
assert actual[0]['assessment']['score']<100
assert actual[1]['assessment']['score']==100

## Bounded optimizer with an injected faulty grader
These two grader fixtures intentionally inflate the first or every draft. The graph, container execution, and deterministic judge use real runtime evidence. The judge is not an LLM. The first case must correct one draft; the second must stop at three refinements and withhold a final grade.

In [ ]:
from tests.test_evaluator_optimizer import LearnsFromJudge,Inflater
optimized=[]
for reasoner in (LearnsFromJudge(),Inflater()):
    result=OrchestratorService(identity,reasoner=reasoner,warm=False).ask(session,{'text':BUGGY})
    optimized.append(result)
    print('GRADER_FAULT_INJECTION_REAL_CONTAINER',type(reasoner).__name__,json.dumps(result,indent=2))
assert optimized[0]['refinement_count']==1
assert optimized[0]['status']=='graded'
assert optimized[1]['refinement_count']==3
assert optimized[1]['grader_calls']==4
assert optimized[1]['status']=='graded_needs_human_review'

In [ ]:
with identity.bind(session):
    record=json.loads(get_learner_profile.invoke({'learner_id':'SPR-1005'}))
assert record['status']=='ok'
print(json.dumps({k:record[k] for k in ('learner_id','completed_modules','python_assignment_score','target_salary_egp','preferred_employers')},indent=2))
print('No paid API requests; business waiting-time improvement is unmeasured.')

## Scope
Passing this notebook establishes local execution and integration for these cases. Python restriction escapes, shared-kernel container vulnerabilities, same-interpreter verdict integrity, production authentication, live LLM decision quality and business waiting-time reduction need separate evaluation. Original media remain unavailable.